“Skills”这个概念（现在官方更常用 **Plugins** 这个词）主要源自 **Microsoft Semantic Kernel (SK)** 框架。

虽然你之前一直在用 OpenAI 原生方式和 LangChain，但了解 SK 的 Skills/Plugins 设计思想对你构建大型企业级 Agent 非常有启发，因为它**把代码和 Prompt 放在了同等地位**。

以下是关于它的来源、原理和上手指南。

---

### 1. 来源：从 "Skills" 到 "Plugins"

#### 起源：Microsoft Semantic Kernel (2023)
当微软开发 Copilot（比如 Office 365 Copilot）时，他们发现需要一种标准方式，让 LLM 既能调用 Python/C# 代码，又能调用写好的 Prompt 模板。他们最初把这种能力包称为 **"Skills"**（技能）。

#### 演变：拥抱 OpenAI 标准
后来 OpenAI 推出了 ChatGPT Plugins 标准。为了统一生态，微软将 "Skills" 重命名为 **"Plugins"**（插件）。
*   **旧称**：Skills (但在代码里的文件夹名、类名中有时还能看到遗留)
*   **新称**：Plugins
*   **核心定义**：一组功能的集合，专门暴露给 AI 使用。

---

### 2. 原理：Prompt 和 代码的“平权”

这是 Semantic Kernel 最独特的设计哲学，与 LangChain 截然不同。

在 SK 中，**一切皆函数 (Function)**。它把能力分为两类，但**用法完全一样**：

1.  **Semantic Functions (语义函数)**:
    *   本质是 **Prompt**。
    *   例如：写一个“翻译函数”，通过 Prompt `把 {{input}} 翻译成英语` 实现。
    *   原理：SK 把 Prompt 包装成一个函数对象，输入是文本，输出是文本。

2.  **Native Functions (原生函数)**:
    *   本质是 **Python/C# 代码**。
    *   例如：`math.sqrt(x)` 或者 `db.query_sql(sql)`。
    *   原理：通过 **装饰器 (Decorators)** 标记函数，让 SK 能扫描到它的描述和参数类型。

#### 为什么要这样设计？
为了**编排 (Orchestration)**。
当你把 Prompt 和 Code 都看作 Function 后，你就可以像搭积木一样串联它们：
*   Step 1 (Native): 从数据库查数据。
*   Step 2 (Semantic): 把数据总结成摘要。
*   Step 3 (Native): 把摘要发送到飞书。

在 SK 的视角里，这只是 `FuncA -> FuncB -> FuncC` 的调用，不需要关心背后是模型在跑还是 CPU 在跑。

---

### 3. 初步使用 (Python 版)

我们需要安装微软的库：
```bash
pip install semantic-kernel
```

下面我带你写一个经典的 **Plugin (Skill)**。我们将创建一个 "MathPlugin"，里面混合包含 **原生代码** 和 **Prompt**。

#### 第一步：定义 Plugin (Skill)

在 Semantic Kernel 中，定义工具不需要像 LangChain 那样继承 BaseTool，只需要写一个普通的 Python 类，加上 `@kernel_function` 装饰器。

```python
import math
from semantic_kernel.functions import kernel_function

class MathPlugin:
    """
    这是一个数学技能包，包含原生计算和语义理解
    """

    # --- 1. Native Function (原生函数：处理硬计算) ---
    @kernel_function(
        description="计算数字的平方根",
        name="Sqrt"
    )
    def sqrt(self, number: float) -> float:
        return math.sqrt(number)

    # --- 2. Native Function (原生函数：加法) ---
    @kernel_function(
        description="计算两个数字的和",
        name="Add"
    )
    def add(self, number1: float, number2: float) -> float:
        return number1 + number2
```

#### 第二步：初始化 Kernel 并加载 Plugin

```python
import asyncio
import os
import dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

# 加载环境变量
dotenv.load_dotenv()

async def main():
    # 1. 初始化 Kernel (内核)
    kernel = Kernel()

    # 2. 配置 AI 服务 (这里用 OpenAI)
    service_id = "default"
    kernel.add_service(
        OpenAIChatCompletion(
            service_id=service_id,
            ai_model_id="gpt-4o-mini",
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    )

    # 3. 导入我们定义的 Plugin
    # plugin_name 类似于以前的 claude_skills Name
    plugin = kernel.add_plugin(MathPlugin(), plugin_name="MathTools")

    print("Plugin 已加载，包含函数:", plugin.functions)

    # 4. 手动调用 Native Function (不经过 LLM，直接调)
    # 这展示了 SK 可以像普通代码一样运行
    result_native = await kernel.invoke(
        plugin["Sqrt"],
        number=16
    )
    print(f"\n[原生调用] Sqrt(16) = {result_native}")

    # 5. 自动调用 (让 LLM 决定调用哪个函数 - 类似 Tool Calling)
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
    from semantic_kernel.contents import ChatHistory

    # 开启自动工具调用
    settings = OpenAIChatPromptExecutionSettings(tool_call_behavior="auto")

    chat_func = kernel.get_prompt_function_from_messages(
        [
            {"role": "system", "content": "你是一个数学助手，遇到计算问题请使用 MathTools。"},
            {"role": "user", "content": "帮我计算 25 的平方根，然后加上 10。"}
        ],
        settings=settings,
        plugin_name="Chat",
        function_name="ChatFunc"
    )

    # 运行
    print("\n[AI 思考中]...")
    result_llm = await kernel.invoke(chat_func)
    print(f"[AI 回答]: {result_llm}")

if __name__ == "__main__":
    asyncio.run(main())
```

### 4. 这里的 "Skills" 和你之前的 "Tools" 有什么区别？

回顾你之前写的 `tools_finance.py`：

1.  **定义方式不同**：
    *   **Native/LangChain**: 你需要手动定义 JSON Schema (`{"type": "function", ...}`)。这很繁琐。
    *   **SK Skills**: 你只需要写 Python 函数并加上 `@kernel_function`，SK 会**自动**帮你生成 JSON Schema 发给 OpenAI。

2.  **组织结构不同**：
    *   **Native**: 工具是散落在列表里的函数 `[func1, func2]`。
    *   **SK Skills**: 工具是**面向对象**的类。`MathPlugin` 类里包含了所有数学相关的方法。这种管理方式在有 100 个工具时会整洁得多。

3.  **Prompt 也是 Skill**：
    *   (上面的代码没展示，但 SK 允许你把一个 `.txt` 文件（里面写着 Prompt）直接加载成一个函数)。
    *   比如 `kernel.import_plugin_from_prompt_directory("./prompts")`，这让管理 Prompt 像管理代码文件一样规范。

### 总结

*   **Skills (Plugins)** 是微软 Semantic Kernel 对“工具”和“能力”的封装标准。
*   它的核心原理是**“装饰器模式”**：用装饰器自动把 Python 代码变成 AI 能读懂的工具描述。
*   相比原生的 OpenAI 写法，使用 Skills 能够让你**少写很多构建 JSON Schema 的样板代码**，并且代码结构更符合工程规范。